# NCES CCD Fiscal Data Cleaning

This section cleans the 2022–23 NCES Common Core of Data (CCD) Local Education Agency Finance Survey (F-33) district-level dataset.

The goal is to:
- inspect the raw district finance data
- identify district and geographic identifiers
- review available fiscal variables
- inspect missing and special-value codes
- select variables relevant to school resources and pediatric mental-health access analysis
- create a cleaned district-level fiscal dataset

In [3]:
import pandas as pd

nces_fiscal_path = "../data/raw/NCES_CC_Fiscal/NCES_CCD_2022_23_Fiscal_F33_District_Data.txt"

nces_fiscal = pd.read_csv(
    nces_fiscal_path,
    sep="\t",
    low_memory=False
)

print("Shape:", nces_fiscal.shape)
nces_fiscal.head()

Shape: (19570, 353)


,LEAID,PID6,UNIT_TYPE,FIPST,CONUM,CSA,CBSA,NAME,STNAME,STABBR,...,FL_AE4B,FL_AE4C,FL_AE4D,FL_AE4E,FL_AE4F,FL_AE4G,FL_AE5,FL_AE6,FL_AE7,FL_AE8
0,0100002,N,N,1,1101,388,33860,Alabama Youth Services,Alabama,AL,...,M,M,M,M,M,M,M,M,M,M
1,0100005,177915,5,1,1095,N,10700,Albertville City,Alabama,AL,...,R,R,R,M,M,M,R,R,R,R
2,0100006,115639,5,1,1095,N,10700,Marshall County,Alabama,AL,...,R,R,R,M,M,M,R,R,R,R
3,0100007,209953,5,1,1073,142,13820,Hoover City,Alabama,AL,...,R,R,R,M,M,M,R,R,R,R
4,0100008,212864,5,1,1089,290,26620,Madison City,Alabama,AL,...,R,R,R,M,M,M,R,R,R,R


In [4]:
print(nces_fiscal.columns.tolist())

['LEAID', 'PID6', 'UNIT_TYPE', 'FIPST', 'CONUM', 'CSA', 'CBSA', 'NAME', 'STNAME', 'STABBR', 'SCHLEV', 'AGCHRT', 'YEAR', 'CCDNF', 'CENFILE', 'GSLO', 'GSHI', 'V33', 'MEMBERSCH', 'TOTALREV', 'TFEDREV', 'C14', 'C15', 'C19', 'C22', 'C23', 'C26', 'C27', 'B11', 'C20', 'C25', 'C36', 'B10', 'B12', 'B14', 'B13', 'TSTREV', 'C01', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C35', 'C38', 'C39', 'TLOCREV', 'T02', 'T06', 'T09', 'T15', 'T40', 'T99', 'D11', 'D23', 'A07', 'A08', 'A09', 'A11', 'A13', 'A15', 'A20', 'A40', 'U11', 'U22', 'U30', 'U50', 'U97', 'C24', 'TOTALEXP', 'TCURELSC', 'TCURINST', 'E13', 'V91', 'V92', 'TCURSSVC', 'E17', 'E07', 'E08', 'E09', 'V40', 'V45', 'V90', 'V85', 'TCUROTH', 'E11', 'V60', 'V65', 'TNONELSE', 'V70', 'V75', 'V80', 'TCAPOUT', 'F12', 'G15', 'K09', 'K10', 'K11', 'L12', 'M12', 'Q11', 'I86', 'Z32', 'Z33', 'Z35', 'Z36', 'Z37', 'Z38', 'V11', 'V13', 'V15', 'V17', 'V21', 'V23', 'V37', 'V29', 'Z34', 'V10', 'V12', 'V14', 'V16', 'V18', 'V22', 'V24', 'V38'

In [5]:
nces_fiscal.info()

<class 'pandas.DataFrame'>
RangeIndex: 19570 entries, 0 to 19569
Columns: 353 entries, LEAID to FL_AE8
dtypes: int64(179), str(174)
memory usage: 52.7 MB


In [6]:
core_cols = [
    "LEAID",
    "NAME",
    "STNAME",
    "STABBR",
    "FIPST",
    "CONUM",
    "YEAR",
    "MEMBERSCH",
    "TOTALREV",
    "TOTALEXP"
]

nces_fiscal[core_cols].head(10)

,LEAID,NAME,STNAME,STABBR,FIPST,CONUM,YEAR,MEMBERSCH,TOTALREV,TOTALEXP
0,0100002,Alabama Youth Services,Alabama,AL,1,1101,23,-2,-1,-1
1,0100005,Albertville City,Alabama,AL,1,1095,23,5900,90930000,81386000
2,0100006,Marshall County,Alabama,AL,1,1095,23,5951,87722000,87755000
3,0100007,Hoover City,Alabama,AL,1,1073,23,13557,229684000,210750000
4,0100008,Madison City,Alabama,AL,1,1089,23,12473,207095000,188825000
5,0100009,Al Inst Deaf And Blind,Alabama,AL,1,1121,23,-1,-2,-2
6,0100010,Al Sch Of Math And Science,Alabama,AL,1,1097,23,-2,-2,-2
7,0100011,Leeds City,Alabama,AL,1,1073,23,2223,33785000,28548000
8,0100012,Boaz City,Alabama,AL,1,1095,23,2471,39406000,35200000
9,0100013,Trussville City,Alabama,AL,1,1073,23,5065,75782000,72155000


In [7]:
nces_fiscal[core_cols].isna().sum()

LEAID        0
NAME         0
STNAME       0
STABBR       0
FIPST        0
CONUM        0
YEAR         0
MEMBERSCH    0
TOTALREV     0
TOTALEXP     0
dtype: int64

In [8]:
special_codes = [-1, -2, -3, -4, -9]

core_fiscal_cols = [
    "MEMBERSCH",
    "TOTALREV",
    "TOTALEXP"
]

for col in core_fiscal_cols:
    print(f"\n{col}")
    print(
        nces_fiscal[col]
        .value_counts()
        .loc[lambda x: x.index.isin(special_codes)]
    )


MEMBERSCH
MEMBERSCH
-2    1039
-3     452
-1      24
-9       3
Name: count, dtype: int64

TOTALREV
TOTALREV
-2    850
-1    588
Name: count, dtype: int64

TOTALEXP
TOTALEXP
-2    850
-1    588
Name: count, dtype: int64


In [9]:
special_membership_rows = nces_fiscal[
    nces_fiscal["MEMBERSCH"].isin([-1, -2, -3, -9])
][
    ["LEAID", "NAME", "STNAME", "UNIT_TYPE", "SCHLEV", "MEMBERSCH", "TOTALREV", "TOTALEXP"]
]

special_membership_rows.head(30)

,LEAID,NAME,STNAME,UNIT_TYPE,SCHLEV,MEMBERSCH,TOTALREV,TOTALEXP
0,0100002,Alabama Youth Services,Alabama,N,N,-2,-1,-1
5,0100009,Al Inst Deaf And Blind,Alabama,N,N,-1,-2,-2
6,0100010,Al Sch Of Math And Science,Alabama,N,N,-2,-2,-2
10,0100018,Alabama School of Fine Arts,Alabama,N,N,-2,-2,-2
16,0100176,JF Ingram State Technical College,Alabama,N,N,-1,-2,-2
34,0100212,Alabama School of Cyber Technology and Enginee...,Alabama,N,N,-2,-2,-2
155,0103583,Covenant Academy of Mobile,Alabama,N,N,-2,-2,-2
156,0103584,Barnabas School of Leadership,Alabama,N,N,-2,-2,-2
235,0400055,EduPreneurship Inc. (4341),Arizona,N,N,-2,-2,-2
240,0400067,Victory High School Inc. (4358),Arizona,N,N,-2,-2,-2


In [10]:
special_membership_rows["MEMBERSCH"].value_counts()

MEMBERSCH
-2    1039
-3     452
-1      24
-9       3
Name: count, dtype: int64

In [11]:
for col in ["UNIT_TYPE", "SCHLEV", "CCDNF"]:
    print(f"\n--- {col} ---")
    print(nces_fiscal[col].value_counts(dropna=False).sort_index())


--- UNIT_TYPE ---
UNIT_TYPE
0       39
1      451
2      219
3      476
5    12902
N     5483
Name: count, dtype: int64

--- SCHLEV ---
SCHLEV
01     4475
02     1263
03    11674
05      294
06      114
07     1060
N       690
Name: count, dtype: int64

--- CCDNF ---
CCDNF
0       40
1    19530
Name: count, dtype: int64


In [12]:
analysis_eligible = nces_fiscal[
    (nces_fiscal["CCDNF"] == 1) &
    (nces_fiscal["MEMBERSCH"] > 0)
]

print("Original rows:", len(nces_fiscal))
print("Eligible rows:", len(analysis_eligible))
print("Excluded rows:", len(nces_fiscal) - len(analysis_eligible))

Original rows: 19570
Eligible rows: 17788
Excluded rows: 1782


In [13]:
nces_fiscal_clean = analysis_eligible.copy()

print("Clean working rows:", len(nces_fiscal_clean))

Clean working rows: 17788


# Select Fiscal Variables for Analysis

The cleaned NCES fiscal dataset contains hundreds of revenue, expenditure, and financial variables.

For this project, variables will be selected based on their relevance to:
- overall district financial resources
- funding source
- instructional spending
- student and support services
- district operating expenditures
- student enrollment
- per-pupil resource measures

In [14]:
candidate_fiscal_cols = [
    # District identifiers
    "LEAID",
    "NAME",
    "STNAME",
    "STABBR",
    "FIPST",
    "CONUM",
    "YEAR",

    # Enrollment
    "MEMBERSCH",

    # Overall revenue
    "TOTALREV",
    "TFEDREV",
    "TSTREV",
    "TLOCREV",

    # Overall expenditures
    "TOTALEXP",
    "TCURELSC",

    # Instruction
    "TCURINST",

    # Support services
    "TCURSSVC",

    # Other current spending
    "TCUROTH",

    # Capital outlay
    "TCAPOUT"
]

nces_fiscal_clean[candidate_fiscal_cols].head()

,LEAID,NAME,STNAME,STABBR,FIPST,CONUM,YEAR,MEMBERSCH,TOTALREV,TFEDREV,TSTREV,TLOCREV,TOTALEXP,TCURELSC,TCURINST,TCURSSVC,TCUROTH,TCAPOUT
1,0100005,Albertville City,Alabama,AL,1,1095,23,5900,90930000,21096000,47555000,22279000,81386000,63879000,37032000,22372000,4475000,15580000
2,0100006,Marshall County,Alabama,AL,1,1095,23,5951,87722000,17968000,48931000,20823000,87755000,78385000,42046000,30337000,6002000,7243000
3,0100007,Hoover City,Alabama,AL,1,1073,23,13557,229684000,11264000,92811000,125609000,210750000,193498000,115447000,67394000,10657000,9775000
4,0100008,Madison City,Alabama,AL,1,1089,23,12473,207095000,11266000,102043000,93786000,188825000,144376000,85185000,53365000,5826000,32405000
7,0100011,Leeds City,Alabama,AL,1,1073,23,2223,33785000,3677000,18266000,11842000,28548000,26499000,15301000,9829000,1369000,612000


In [16]:
fiscal_measure_cols = [
    "TOTALREV",
    "TFEDREV",
    "TSTREV",
    "TLOCREV",
    "TOTALEXP",
    "TCURELSC",
    "TCURINST",
    "TCURSSVC",
    "TCUROTH",
    "TCAPOUT"
]

special_codes = [-1, -2, -3, -4, -9]

for col in fiscal_measure_cols:
    counts = (
        nces_fiscal_clean[col]
        .value_counts()
        .loc[lambda x: x.index.isin(special_codes)]
    )

    print(f"\n{col}")
    print(counts if not counts.empty else "No special codes")


TOTALREV
TOTALREV
-1    247
-2    133
Name: count, dtype: int64

TFEDREV
TFEDREV
-1    247
-2    133
Name: count, dtype: int64

TSTREV
TSTREV
-1    247
-2    133
Name: count, dtype: int64

TLOCREV
TLOCREV
-1    247
-2    133
Name: count, dtype: int64

TOTALEXP
TOTALEXP
-1    247
-2    133
Name: count, dtype: int64

TCURELSC
TCURELSC
-1    247
-2    133
Name: count, dtype: int64

TCURINST
TCURINST
-1    247
-2    133
Name: count, dtype: int64

TCURSSVC
TCURSSVC
-1    247
-2    133
Name: count, dtype: int64

TCUROTH
TCUROTH
-1    247
-2    133
Name: count, dtype: int64

TCAPOUT
TCAPOUT
-1    247
-2    133
Name: count, dtype: int64


In [17]:
import numpy as np

nces_fiscal_clean[fiscal_measure_cols] = (
    nces_fiscal_clean[fiscal_measure_cols]
    .replace([-1, -2, -3, -4, -9], np.nan)
)

nces_fiscal_clean[fiscal_measure_cols].isna().sum()

TOTALREV    380
TFEDREV     380
TSTREV      380
TLOCREV     380
TOTALEXP    380
TCURELSC    380
TCURINST    380
TCURSSVC    380
TCUROTH     380
TCAPOUT     380
dtype: int64

In [18]:
for col in fiscal_measure_cols:
    negative_count = (nces_fiscal_clean[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

TOTALREV: 0 negative values
TFEDREV: 0 negative values
TSTREV: 0 negative values
TLOCREV: 0 negative values
TOTALEXP: 0 negative values
TCURELSC: 0 negative values
TCURINST: 0 negative values
TCURSSVC: 0 negative values
TCUROTH: 0 negative values
TCAPOUT: 0 negative values


# Create Per-Pupil Fiscal Measures

Per-pupil measures standardize district finances by student enrollment, allowing districts of different sizes to be compared more meaningfully.

In [19]:
nces_fiscal_clean["REV_PER_PUPIL"] = (
    nces_fiscal_clean["TOTALREV"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["EXP_PER_PUPIL"] = (
    nces_fiscal_clean["TOTALEXP"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["INST_PER_PUPIL"] = (
    nces_fiscal_clean["TCURINST"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["SUPPORT_PER_PUPIL"] = (
    nces_fiscal_clean["TCURSSVC"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["FEDREV_PER_PUPIL"] = (
    nces_fiscal_clean["TFEDREV"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["STATEREV_PER_PUPIL"] = (
    nces_fiscal_clean["TSTREV"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["LOCALREV_PER_PUPIL"] = (
    nces_fiscal_clean["TLOCREV"] / nces_fiscal_clean["MEMBERSCH"]
)

In [20]:
per_pupil_cols = [
    "REV_PER_PUPIL",
    "EXP_PER_PUPIL",
    "INST_PER_PUPIL",
    "SUPPORT_PER_PUPIL",
    "FEDREV_PER_PUPIL",
    "STATEREV_PER_PUPIL",
    "LOCALREV_PER_PUPIL"
]

nces_fiscal_clean[per_pupil_cols].describe().round(2)

,REV_PER_PUPIL,EXP_PER_PUPIL,INST_PER_PUPIL,SUPPORT_PER_PUPIL,FEDREV_PER_PUPIL,STATEREV_PER_PUPIL,LOCALREV_PER_PUPIL
count,17408.00,17408.00,17408.00,17408.00,17408.00,17408.00,17408.00
mean,32293.58,31747.74,13252.72,10522.37,3835.01,12629.59,15828.98
std,229960.75,255751.18,65255.61,54661.09,18815.26,44060.22,189710.22
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,15324.19,14891.83,7362.68,4704.47,1267.44,6387.03,3560.36
50%,18848.55,18430.20,8986.34,6040.19,2124.20,8971.56,6577.00
75%,24718.58,24538.61,11697.93,8275.23,3359.21,11911.88,11906.88
max,17590000.00,23970000.00,4804000.00,3323000.00,1156000.00,2271000.00,15134000.00


In [21]:
support_detail_cols = [
    "E17",
    "E07",
    "E08",
    "E09",
    "V40",
    "V45",
    "V90"
]

nces_fiscal_clean[support_detail_cols].describe()

,E17,E07,E08,E09,V40,V45,V90
count,1.778800e+04,1.778800e+04,1.778800e+04,1.778800e+04,1.778800e+04,1.778800e+04,1.778800e+04
mean,2.992477e+06,2.366917e+06,8.822916e+05,2.506965e+06,4.326287e+06,1.853231e+06,1.814554e+06
std,1.116951e+07,1.166584e+07,2.267764e+06,1.352676e+07,2.568176e+07,1.388603e+07,1.245450e+07
min,-2.000000e+00,-2.000000e+00,-2.000000e+00,-2.000000e+00,-2.000000e+00,-2.000000e+00,-2.000000e+00
25%,1.710000e+05,1.060000e+05,1.900000e+05,2.660000e+05,4.760000e+05,8.900000e+04,1.420000e+05
50%,5.830000e+05,4.440000e+05,4.340000e+05,6.900000e+05,1.242500e+06,4.230000e+05,4.475000e+05
75%,2.176250e+06,1.629500e+06,9.230000e+05,1.853000e+06,3.390250e+06,1.471250e+06,1.242000e+06
max,6.554570e+08,9.666850e+08,1.428910e+08,1.326779e+09,2.679139e+09,1.682409e+09,1.385875e+09


In [22]:
support_detail_cols = [
    "E17",
    "E07",
    "E08",
    "E09",
    "V40",
    "V45",
    "V90"
]

nces_fiscal_clean[support_detail_cols] = (
    nces_fiscal_clean[support_detail_cols]
    .replace([-1, -2, -3, -4, -9], np.nan)
)

nces_fiscal_clean[support_detail_cols].isna().sum()

E17    380
E07    380
E08    380
E09    380
V40    380
V45    380
V90    380
dtype: int64

In [23]:
nces_fiscal_clean["STUDENT_SUPPORT_PER_PUPIL"] = (
    nces_fiscal_clean["E17"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["INST_STAFF_SUPPORT_PER_PUPIL"] = (
    nces_fiscal_clean["E07"] / nces_fiscal_clean["MEMBERSCH"]
)

nces_fiscal_clean["SCHOOL_ADMIN_PER_PUPIL"] = (
    nces_fiscal_clean["E09"] / nces_fiscal_clean["MEMBERSCH"]
)

In [24]:
nces_fiscal_clean["TRANSPORT_PER_PUPIL"] = (
    nces_fiscal_clean["V45"] / nces_fiscal_clean["MEMBERSCH"]
)

In [25]:
nces_fiscal_clean[
    ["LEAID", "NAME", "STNAME", "MEMBERSCH", "REV_PER_PUPIL", "EXP_PER_PUPIL"]
].sort_values("REV_PER_PUPIL", ascending=False).head(20)

,LEAID,NAME,STNAME,MEMBERSCH,REV_PER_PUPIL,EXP_PER_PUPIL
16275,4280210,Erie County Technical School,Pennsylvania,1,1.759000e+07,2.397000e+07
11800,3480240,Monmouth-Ocean Educational Services Commission...,New Jersey,6,1.431867e+07,1.444183e+07
16277,4280230,Franklin County CTC,Pennsylvania,1,1.115100e+07,1.069500e+07
11806,3480360,Hunterdon County Educational Services Commission,New Jersey,2,8.176000e+06,7.957500e+06
15716,4200878,Mifflin County Academy of Science and Technology,Pennsylvania,1,5.666000e+06,4.925000e+06
15340,4100025,Douglas ESD,Oregon,11,4.025909e+06,3.533727e+06
3318,0691011,Inyo County Office of Education,California,5,3.962400e+06,3.829000e+06
16294,4280430,Venango Technology Center,Pennsylvania,2,3.954500e+06,3.914000e+06
15926,4210970,Greater Johnstown CTC,Pennsylvania,3,2.997667e+06,2.779333e+06
16289,4280360,Western Montgomery CTC,Pennsylvania,3,2.823667e+06,2.704000e+06
